# Prompt Strategy Lab - Walkthrough

The original exercise notebook from FAU's **COT 6930** (Generative AI & Software Development Lifecycles) course, cleaned for portfolio use. Compares Chain-of-Thought, Reflection, Few-Shot, Self-Consistency and other prompt-engineering strategies on the same task using a local TinyLLaMA via Ollama.

This is the **complete, exploratory** version with all the original model outputs preserved. For a shorter, package-using demo with a clean Python API, see [`demo.ipynb`](demo.ipynb). For the rest of the project (library, tests, CLI), see the [project README](../README.md).


### Selected Task: Problem Identification

**User Prompt:**

In [ ]:
# --- Step 1: Define raw User Prompt ---
USER_PROMPT = "Help me brainstorm problems for an AI-powered study group matching platform."
print("User Prompt:\n", USER_PROMPT)

**Meta-Prompt (Chain-of-Thought):**

In [ ]:
# --- Step 2: Apply a Prompt Augmentation Technique ---
META_PROMPT = f"""
You are a product manager focused on educational technology.
Your goal is to help the user identify key problems related to their proposed solution.
Think step by step about the user's request and potential problems before answering.

Task: Brainstorm problem statements for this idea:
{USER_PROMPT}

Process:
1. Analyze the user's request and the target solution (AI-powered study group matching platform).
2. Consider the different types of users who would use this platform (students, potentially instructors).
3. Think about the challenges and pain points these users face when trying to form or participate in study groups.
4. Brainstorm potential problems that an AI-powered platform could help solve in the context of study groups.
5. For each potential problem, identify the user pain point, any assumptions made, and a clarifying question.
6. List at least 5 problem statements based on this analysis.

List at least 5 problem statements, each with:
- The user pain point
- Assumptions made
- A clarifying question
""".strip()
print("Meta-Prompt:\n", META_PROPT)

**Explanation of Technique:**

I chose the **Chain-of-Thought** prompting technique for the meta-prompt. This technique instructs the model to break down the problem into intermediate steps and explicitly show its reasoning process. By guiding the model through analyzing the request, considering users and their pain points, brainstorming problems, and then structuring the output with specific elements (pain point, assumptions, question), it encourages a more thorough and relevant set of problem statements.

In [ ]:
# We will be working with `tinyllama`
# https://ollama.com/library/tinyllama
#
# INSTALL ANY OTHER MODEL YOU WANT TO TRY OUT HERE!
#
!ollama pull tinyllama

In [ ]:
# This will list all installed models
!ollama list

NAME                ID              SIZE      MODIFIED       
tinyllama:latest    2644915ede35    637 MB    11 seconds ago    


In [ ]:
##
# Install the Ollama Client
!pip install ollama
from ollama import Client

In [ ]:
##
# Create the re-usable function ``model_request(.)``
from ollama import Client

# Default model
DEFAULT_MODEL = "tinyllama"

# Ollama client
client = Client()

def model_request(
    prompt,
    model=DEFAULT_MODEL,
    temperature=0.7,
    top_k=40,
    top_p=0.9,
    num_predict=200,
    context_window=2048
):
    """
    Call the model client with configurable parameters.

    Args:
        prompt (str | list[str]): The user prompt(s).
        model (str): Model name to call (defaults to DEFAULT_MODEL).
        temperature (float): Controls randomness (higher = more random).
        top_k (int): Number of candidates considered at each step.
        top_p (float): Nucleus sampling threshold.
        num_predict (int): Maximum number of tokens to generate.
        context_window (int): Max number of tokens considered from prior conversation.

    Returns:
        tuple: (model's response text, number of tokens used).
    """

    # Wrap prompt(s) into messages
    messages = [{"role": "user", "content": p} for p in ([prompt] if isinstance(prompt, str) else prompt)]

    response = client.chat(
        model=model,
        messages=messages,
        options={
            "temperature": temperature,
            "top_k": top_k,
            "top_p": top_p,
            "num_predict": num_predict,
            "num_ctx": context_window
        }
    )

    text = response["message"]["content"]
    tokens_used = response.get("eval_count", len(text.split()))

    return text, tokens_used


---
# (PART II) EXPERIMENTS

### Experiment 1 - Prompt Augmentation for Problem Ideation

Simulate a user talking to the Solution Design Bot for help with Problem Ideation about a product. Each variation uses a different prompt design technique from the fundamentals document.

Remember that you are creating Meta-Prompts for Prompt Augmentation:

```
User (Prompt)->
  Bot ->
  AI Pipeline ->
  Prompt Augmentation (Meta-Prompts, Augmented Prompt) ->
  Model Request (Augmentated Prompt, Parameters)
```

In [ ]:
# --- Step 1: Define raw User Prompt ---
# NOTE: REPLACE the 'target solution' with your proposed project!

USER_PROMPT = "Help me brainstorm problems for an AI-powered exam studying app for SAT, GRE, and Medical Exams"
#USER_PROMPT = "Help me brainstorm problems for an AI-powered {{YOUR SOLUTION HERE}}
print("User Prompt:\n", USER_PROMPT)

User Prompt:
 Help me brainstorm problems for an AI-powered exam studying app for SAT, GRE, and Medical Exams


In [ ]:
# --- Step 2: Apply a Prompt Augmentation Technique ---
# Choose ONE technique (Persona / Reflection / Alternative Approaches)

# Example: Chain of Thought Prompting
META_PROMPT = f"""
You are a senior product strategist.
Think step by step about the user's request and potential problems before answering.

Task: Help a team brainstorm problem statements for this idea:
{USER_PROMPT}

Process:
1. Analyze the user's request and the target solution (AI-powered exam studying app for SAT, GRE, and Medical Exams).
2. Consider the different types of users who would use this app (students preparing for various exams).
3. Think about the challenges and pain points these students face when studying for exams.
4. Brainstorm potential problems that an AI-powered app could help solve.
5. For each potential problem, identify the user pain point, any assumptions made, and a clarifying question.
6. List at least 5 problem statements based on this analysis.

List at least 5 problem statements, each with:
- The user pain point
- Assumptions made
- A clarifying question
""".strip()

In [ ]:
# Example: Reflection Prompting
# META_PROMPT = f"""
# Generate 5 problem statements for:
# {USER_PROMPT}
#
# Then review your own output:
# - Identify weaknesses, gaps, or biases
# - Suggest 2 improvements or alternative framings
# """.strip()

In [ ]:
# Example: Alternative Approaches Prompting
# META_PROMPT = f"""
# Frame problems for:
# {USER_PROMPT}
#
# Produce three alternative perspectives:
# 1. From students
# 2. From instructors
# 3. From administrators
#
# For each, list 2–3 problem statements.
# """.strip()

In [ ]:
# --- Step 3: Construct Augmented Prompt ---
AUGMENTED_PROMPT = META_PROMPT
print("\n--- Augmented Prompt ---\n")
print(AUGMENTED_PROMPT)


--- Augmented Prompt ---

You are a senior product strategist.
Think step by step about the user's request and potential problems before answering.

Task: Help a team brainstorm problem statements for this idea:
Help me brainstorm problems for an AI-powered exam studying app for SAT, GRE, and Medical Exams

Process:
1. Analyze the user's request and the target solution (AI-powered exam studying app for SAT, GRE, and Medical Exams).
2. Consider the different types of users who would use this app (students preparing for various exams).
3. Think about the challenges and pain points these students face when studying for exams.
4. Brainstorm potential problems that an AI-powered app could help solve.
5. For each potential problem, identify the user pain point, any assumptions made, and a clarifying question.
6. List at least 5 problem statements based on this analysis.

List at least 5 problem statements, each with:
- The user pain point
- Assumptions made
- A clarifying question


In [ ]:
# --- Step 4: Call the model with all parameters explicitly set ---
response, tokens = model_request(
    prompt=AUGMENTED_PROMPT,
    model=DEFAULT_MODEL,   # or override with a different model name
    temperature=0.7,       # controls randomness (0=deterministic, >1 = more random)
    top_k=40,              # number of candidates considered at each step
    top_p=0.9,             # nucleus sampling threshold
    num_predict=1000,      # max tokens to generate
    context_window=2048    # size of context window
)

print("\n--- Bot Response ---\n")
print(response)
print(f"\n[Tokens used: {tokens}]")


--- Bot Response ---

1. User Pain Point: I don't have enough time to study for SAST exam due to my busy schedule.
Assumptions Made: - The app is designed to be user-friendly and does not require a lot of time to use. - Students can prioritize their time according to their schedules.
Clarifying Question: How can the app help students save time and reduce stress during SAST exam prep? Answer: Users could take practice exams with the app, as it provides comprehensive information on different SAST topics. By doing so, they can review the material more efficiently and save time while preparing for the actual exam.
2. User Pain Point: I'm not comfortable writing down notes in a book or taking notes during class due to my anxiety.
Assumptions Made: - The app is user-friendly and does not require any prior knowledge of the subject being studied. - Students can take notes during class, as it's easy to use and doesn't require any specialized skills.
Clarifying Question: How can the app help st

### Experiment 2 - Prompt Augmentation for Solution Ideation

In [ ]:
# --- Step 1: Define raw User Prompt ---
# NOTE: REPLACE the 'target solution' with your proposed project!
USER_PROMPT = "Help me brainstorm problems for an AI-powered exam studying app for SAT, GRE, and Medical Exams"
#USER_PROMPT = "Help me brainstorm problems for an AI-powered {{YOUR SOLUTION HERE}}
print("User Prompt:\n", USER_PROMPT)

User Prompt:
 Help me brainstorm problems for an AI-powered exam studying app for SAT, GRE, and Medical Exams


In [ ]:
# --- Step 2: Apply a Prompt Augmentation Technique ---

# Example: Chain of Thought Prompting
META_PROMPT = f"""
You are a Solution Design Bot with expertise in AI-powered educational tools.
Think step by step about potential solutions before presenting the final concepts.

Task: Generate 2–3 detailed solution concepts for this idea:
{USER_PROMPT}

Process:
1. Analyze the user's prompt and the target problem space (AI-powered exam studying app for SAT, GRE, and Medical Exams).
2. Consider the key features and functionalities an AI could provide in this context (e.g., personalized study plans, practice questions, performance analysis, content explanation).
3. Brainstorm 3 distinct approaches to building such an app, focusing on different core AI capabilities or user experiences.
4. For each approach, outline the core concept, key features, potential risks or challenges in implementation, and measurable success metrics.
5. Present the 2-3 solution concepts clearly.

Generate 2–3 detailed solution concepts, each including:
- **Core Concept:** A brief summary of the main idea.
- **Key Features:** List the essential functionalities powered by AI.
- **Potential Risks/Challenges:** Identify potential difficulties in development or adoption.
- **Success Metrics:** Define how the success of this solution would be measured.
""".strip()

In [ ]:
# Example: Template Prompting
#META_PROMPT = f"""
#You are a Solution Design Bot.
#
#Task: Produce a structured **one-pager solution concept** for this idea:
#{USER_PROMPT}
#
#Use the following template:
#Problem Summary:
#Solution Overview:
#Core Features:
#Differentiators:
#Risks & Open Questions:
#Success Metrics:
#""".strip()

In [ ]:
# Example: Comparative Prompting
#META_PROMPT = f"""
#You are a Solution Design Bot.
#
#Task: Generate and compare **three different solution approaches** for this idea:
#{USER_PROMPT}
#
#For each approach, include:
#- Core idea
#- Key benefits
#- Trade-offs
#- Risk factors
#
#Then conclude with a recommendation: which approach is most viable and why?
#""".strip()

In [ ]:
# --- Step 3: Construct Augmented Prompt ---
AUGMENTED_PROMPT = META_PROMPT
print("\n--- Augmented Prompt ---\n")
print(AUGMENTED_PROMPT)


--- Augmented Prompt ---

You are a Solution Design Bot with expertise in AI-powered educational tools.
Think step by step about potential solutions before presenting the final concepts.

Task: Generate 2–3 detailed solution concepts for this idea:
Help me brainstorm problems for an AI-powered exam studying app for SAT, GRE, and Medical Exams

Process:
1. Analyze the user's prompt and the target problem space (AI-powered exam studying app for SAT, GRE, and Medical Exams).
2. Consider the key features and functionalities an AI could provide in this context (e.g., personalized study plans, practice questions, performance analysis, content explanation).
3. Brainstorm 3 distinct approaches to building such an app, focusing on different core AI capabilities or user experiences.
4. For each approach, outline the core concept, key features, potential risks or challenges in implementation, and measurable success metrics.
5. Present the 2-3 solution concepts clearly.

Generate 2–3 detailed sol

In [ ]:
# --- Step 4: Call the model with all parameters explicitly set ---
# NOTE: TEST WITH VARIATIONS OF MODEL PARAMTERS; FIND THE CONFIGURATION THAT DELIVERS THE BEST OUTPUT
response, tokens = model_request(
    prompt=AUGMENTED_PROMPT,
    model=DEFAULT_MODEL,   # or override with a different model name
    temperature=0.6,       # controls randomness (0=deterministic, >1 = more random) - Slightly reduced for focus
    top_k=40,              # number of candidates considered at each step
    top_p=0.8,             # nucleus sampling threshold - Slightly reduced for focus
    num_predict=1000,      # max tokens to generate
    context_window=2048    # size of context window
)

print("\n--- Bot Response ---\n")
print(response)
print(f"\n[Tokens used: {tokens}]")


--- Bot Response ---

1. **AI-powered Exam Study App for SAHAM:**
   - Core Concept: AI-powered exam study app to provide personalized study plans, practice questions, performance analysis, and content explanation.
   - Key Features:
     - AI-based algorithm to recommend study topics based on user history and test data.
     - Automated grading system for individualized feedback on performance.
     - Real-time feedback and reminders during exams.
     - Integration with educational institutions and tutors for personalized support and guidance.
   - Potential Risks/Challenges:
     - Limited data availability or accuracy of test data.
     - User privacy concerns over AI-powered algorithms.
   - Success Metrics:
     - Increased exam success rates and improved student outcomes.
     - Improved engagement and retention rates among students.

2. **AI-Powered Exam Study App for Medical Exams:**
   - Core Concept: AI-powered exam study app to provide personalized study plans, practice qu

### Experiment 3 - Prompt Augmentation for Requirement Analysis

In [ ]:
# --- Step 1: Define raw User Prompt ---
# NOTE: REPLACE the 'target solution' with your proposed project!
USER_PROMPT = "Help me write requirements for an AI-powered exam studying app for SAT, GRE, and Medical Exams"
#USER_PROMPT = "Help me write requirements for an AI-powered {{YOUR SOLUTION HERE}}"
print("User Prompt:\n", USER_PROMPT)


User Prompt:
 Help me write requirements for an AI-powered exam studying app for SAT, GRE, and Medical Exams


In [ ]:
# --- Step 2: Apply a Prompt Augmentation Technique ---
# Choose ONE technique (Template / FactCheck / Reflection)

# Example: Template Prompting
META_PROMPT = f"""
You are a Requirements Engineer Bot.

Task: Write **user stories with acceptance criteria** for this idea:
{USER_PROMPT}

Use the following template for each story:
User Story: As a <role>, I want <capability>, so that <benefit>.
Acceptance Criteria:
- Given ...
- When ...
- Then ...
Notes:
""".strip()

In [ ]:
# Example: Fact Check List Prompting
#META_PROMPT = f"""
#You are a Requirements Engineer Bot.

#Task: Write requirements for this idea:
#{USER_PROMPT}

#For each requirement:
#1. Provide the requirement statement.
#2. List factual claims (that can be verified).
#3. List assumptions (that need validation).
#4. Suggest one clarifying question.

#Format:
#Requirement: ...
#Facts: ...
#Assumptions: ...
#Question: ...
#""".strip()

In [ ]:
# Example: Reflection Prompting
#META_PROMPT = f"""
#You are a Requirements Engineer Bot.
#
#Task: Write requirements for this idea:
#{USER_PROMPT}
#
#Step 1: Generate 5–7 functional and non-functional requirements.
#Step 2: Reflect on your own output:
#- Are all requirements testable?
#- Did you miss any critical constraints (e.g., privacy, accessibility, reliability)?
#- Rewrite or add 2 improved requirements if needed.
#
#Output format:
#Requirements:
#1) ...
#2) ...
#3) ...
#Reflection:
#- gaps_or_risks: ...
#- improvements:
#   A) ...
#   B) ...
#""".strip()

# (Part III) EXERCISE - PROMPT AUGMENTATION FOR LATER SDLC PHASES

<img src="http://generativeintelligencelab.ai/images/sdlc-info.png" width="700">

* **System Design**: translate requirements into system architecture
* **Solution Development**: adopt best practices for coding; generate code structure.

#### Challenge:

* Select 3 Prompt Engineering techniques below (check class presentation for details)
* Apply them to these phases.

1. Zero-Shot Prompting
1. Few-Shot Prompting
1. Chain-of-Thought (CoT)
1. Meta Prompting
1. Self-Consistency
1. Generated Knowledge Prompting
1. Prompt Chaining
1. Tree of Thoughts (ToT)
1. Automatic Reasoning
1. Automatic Prompt Engineering (APE)
1. Active-Prompt
1. Directional Stimulus Prompting
1. Reflexion
1. Graph Prompting

## System Design

In [ ]:
# --- Step 1: Define raw User Prompt ---
# Note: ADJUST for your solution scenario

USER_PROMPT = "Propose a backend code structure for an AI-powered exam studying app for SAT, GRE, and Medical Exams"
print("User Prompt:\n", USER_PROMPT)


User Prompt:
 Propose a backend code structure for an AI-powered exam studying app for SAT, GRE, and Medical Exams


In [ ]:
# --- Step 2: Apply a Prompt Augmentation Technique ---
# Technique: Few-Shot Prompting
# NOTE: PICK ANOTHER TECHNIQUE AND COMPLETE THE BELOW

META_PROMPT = f"""
You are a Solution Design Bot.

Here are examples of backend project structures:

Example :
project/
├── app/
│   ├── __init__.py
│   ├── models.py
│   ├── routes.py
│   └── services.py
├── tests/
│   └── test_app.py
├── requirements.txt
└── run.py

Now, using a similar style, propose a **backend code structure** for:
👉 {USER_PROMPT}

Include:
- Folder layout
- Key files
- A short description of each component
""".strip()

In [ ]:
# Alternative 1
# Combining chain-of-thought and reflection
META_PROMPT_2 = f"""
You are a Solution Design Bot.

Goal
Create a clear and production ready backend structure for the request below.
User request
{USER_PROMPT}

How to think
1 Do all deep reasoning silently. Do not reveal your chain of thought.
2 Work in two internal stages
   2a Plan stage
      i unpack the problem into subparts
      ii consider two or three plausible designs
      iii choose one primary design with tradeoffs noted
   2b Reflection stage
      i verify that the output format is complete
      ii search for missing pieces or contradictions
      iii correct any gaps you find

What to output
1 Folder layout as a tree using simple indentation
2 Key files with one line purpose each
3 Core modules and their responsibilities
4 API surface
   4a main routes with method and short purpose
   4b auth and rate limits if relevant
5 Data layer
   5a models and relations
   5b storage choice and rationale in one or two lines
6 Services and background jobs
7 Config logging and observability
8 Testing plan and minimal CI notes
9 Deployment notes and scalability tips
10 Assumptions open questions and risks with quick mitigations

Style rules for your answer
1 Be concise and specific
2 Use clear section headers
3 Do not print any hidden notes or step by step reasoning
4 Prefer short sentences
5 Use neutral and professional tone

Quality checklist to run silently before you answer
1 Does the layout map to every requirement in the user request
2 Are security concerns covered auth input validation secrets and PII
3 Are failure cases and retries mentioned where needed
4 Is the plan feasible for a small first release and extendable later
5 Are tests observability and deployment addressed

Answer now with the final structured result only.
""".strip()


In [ ]:
# Alternative 2
META_PROMPT_3 = f"""
You are a Solution Design Bot.

# Technique 1: Few-Shot Prompting
Here are examples of backend project structures:

Example A:
project/
├── app/
│   ├── __init__.py
│   ├── models.py
│   ├── routes.py
│   ├── services.py
├── tests/
│   └── test_app.py
├── requirements.txt
└── run.py

Example B:
backend/
├── src/
│   ├── api/
│   │   ├── controllers.py
│   │   ├── serializers.py
│   ├── core/
│   │   ├── config.py
│   │   ├── database.py
│   ├── features/
│   │   ├── users.py
│   │   ├── exams.py
│   │   ├── ai_engine.py
├── tests/
│   └── unit/
│   └── integration/
└── pyproject.toml

# Technique 2: Chain-of-Thought with Reflection
Think through the problem silently in two passes:
1. Planning stage:
   - Break down the problem into subparts (data layer, APIs, services, testing, deployment).
   - Consider at least two design options.
   - Select the most practical and extensible option.
2. Reflection stage:
   - Verify that each key part of the backend is covered (APIs, models, services, security, tests, observability).
   - Check for consistency, scalability, and feasibility for a first release.
   - Fix missing components or contradictions before finalizing.

# Technique 3: Self-Consistency
- Generate at least two candidate backend structures.
- Internally compare them.
- Output only the **final best-consistent structure**.

# Output Requirements
For the user request below, propose a backend code structure that includes:
1. Folder layout (tree format).
2. Key files and short descriptions.
3. API routes (with method and purpose).
4. Data models and storage choice (with short rationale).
5. Services and background jobs.
6. Config, logging, and observability.
7. Testing strategy and CI notes.
8. Deployment and scalability notes.
9. Assumptions, open questions, and risks with quick mitigations.

User Request:
👉 {USER_PROMPT}

Final Answer:
Provide only the consolidated best backend code structure in a clean, professional format.
""".strip()

In [ ]:
# --- Step 3: Construct Augmented Prompt ---
AUGMENTED_PROMPT = META_PROMPT
print("\n--- Augmented Prompt ---\n")
print(AUGMENTED_PROMPT)


--- Augmented Prompt ---

You are a Solution Design Bot.

Here are examples of backend project structures:

Example :
project/
├── app/
│   ├── __init__.py
│   ├── models.py
│   ├── routes.py
│   └── services.py
├── tests/
│   └── test_app.py
├── requirements.txt
└── run.py

Now, using a similar style, propose a **backend code structure** for:
👉 Propose a backend code structure for an AI-powered exam studying app for SAT, GRE, and Medical Exams

Include:
- Folder layout
- Key files
- A short description of each component


In [ ]:
AUGMENTED_PROMPT_2 = META_PROMPT_2
print("\n--- Augmented Prompt 2---\n")
print(AUGMENTED_PROMPT_2)


--- Augmented Prompt 2---

You are a Solution Design Bot.

Goal
Create a clear and production ready backend structure for the request below.
User request
Propose a backend code structure for an AI-powered exam studying app for SAT, GRE, and Medical Exams

How to think
1 Do all deep reasoning silently. Do not reveal your chain of thought.
2 Work in two internal stages
   2a Plan stage
      i unpack the problem into subparts
      ii consider two or three plausible designs
      iii choose one primary design with tradeoffs noted
   2b Reflection stage
      i verify that the output format is complete
      ii search for missing pieces or contradictions
      iii correct any gaps you find

What to output
1 Folder layout as a tree using simple indentation
2 Key files with one line purpose each
3 Core modules and their responsibilities
4 API surface
   4a main routes with method and short purpose
   4b auth and rate limits if relevant
5 Data layer
   5a models and relations
   5b storage 

In [ ]:
AUGMENTED_PROMPT_3 = META_PROMPT_3
print("\n--- Augmented Prompt 3---\n")
print(AUGMENTED_PROMPT_3)


--- Augmented Prompt 3---

You are a Solution Design Bot.

# Technique 1: Few-Shot Prompting
Here are examples of backend project structures:

Example A:
project/
├── app/
│   ├── __init__.py
│   ├── models.py
│   ├── routes.py
│   ├── services.py
├── tests/
│   └── test_app.py
├── requirements.txt
└── run.py

Example B:
backend/
├── src/
│   ├── api/
│   │   ├── controllers.py
│   │   ├── serializers.py
│   ├── core/
│   │   ├── config.py
│   │   ├── database.py
│   ├── features/
│   │   ├── users.py
│   │   ├── exams.py
│   │   ├── ai_engine.py
├── tests/
│   └── unit/
│   └── integration/
└── pyproject.toml

# Technique 2: Chain-of-Thought with Reflection
Think through the problem silently in two passes:
1. Planning stage:
   - Break down the problem into subparts (data layer, APIs, services, testing, deployment).
   - Consider at least two design options.
   - Select the most practical and extensible option.
2. Reflection stage:
   - Verify that each key part of the backend is cov

In [ ]:
# --- Step 4: Call the model with all parameters explicitly set ---
# NOTE: TEST WITH VARIATIONS OF MODEL PARAMTERS; FIND THE CONFIGURATION THAT DELIVERS THE BEST OUTPUT
response, tokens = model_request(
    prompt=AUGMENTED_PROMPT,
    model=DEFAULT_MODEL,   # override if desired
    temperature=0.7,
    top_k=40,
    top_p=0.9,
    num_predict=600,
    context_window=2048
)

print("\n--- Bot Response ---\n")
print(response)
print(f"\n[Tokens used: {tokens}]")


--- Bot Response ---

Folders Layout:

👉 Folder Name: `backend`
    - 1. `models`: Contains models for the application's database, including schema definition and entity classes. (Examples: Question, Answer, Score, User)
    - 2. `routes`: Contains routes and endpoints that map HTTP requests to RESTful actions. (Examples: GET, POST, PUT, DELETE, PATCH)
    - 3. `services`: This folder contains services used by the application, such as caching, database operations, etc. (Examples: Cache middleware, ORM layer, Database driver, etc.)
    - 4. `tests`: Contains unit tests for the application's core components, including endpoints and services.

👉 Key Files:

- `app.py`: This file is the primary entry point of your backend project. It provides an interface to interact with different components. (Examples: Configure database, middleware, etc.)
    - Import all necessary dependencies from other files.
        - Use decorators like `@login_required` and `@user_passes_csrf` to validate user au

In [ ]:
response, tokens = model_request(
    prompt=AUGMENTED_PROMPT_2,
    model=DEFAULT_MODEL,   # override if desired
    temperature=0.7,
    top_k=40,
    top_p=0.9,
    num_predict=600,
    context_window=2048
)

print("\n--- Bot Response ---\n")
print(response)
print(f"\n[Tokens used: {tokens}]")


--- Bot Response ---

To create a clear and production-ready backend structure for the request below, we propose the following solution:

1. Do all deep reasoning silently by unpacking the problem into subparts, considering two or three plausible designs, and choosing one primary design with tradeoffs noted. We aim to plan and work in two internal stages:
   2a. Plan stage: Unpacking the problem into subparts
       i. Unpacking the problem into sub-problems, each with a clear and feasible solution
       ii. Comparing and selecting the best option based on tradeoffs (e.g. Performance vs. Resources)
   2b. Reflection stage: Verifying output format completeness
       i. Identifying missing pieces or contradictions in the output format, and correcting them if necessary
       ii. Searching for missing pieces or contradiction in the API surface, and correcting them if necessary
3. Core modules and their responsibilites:
   3a. Models and relations: Designing appropriate data models and 

In [ ]:
response, tokens = model_request(
    prompt=AUGMENTED_PROMPT_3,
    model=DEFAULT_MODEL,   # override if desired
    temperature=0.7,
    top_k=40,
    top_p=0.9,
    num_predict=600,
    context_window=2048
)

print("\n--- Bot Response ---\n")
print(response)
print(f"\n[Tokens used: {tokens}]")


--- Bot Response ---

Here's an example of how to implement the Technique 3: Self-Consistency with the Solution Design Bot:

# Technique 3: Self-Consistency
- Generate at least two candidate backend structures:
  - A backend for SAT, GRE, and Medical Exams
  - A backend for a different purpose (e.g., a web app)
- Internally compare the two structures to identify gaps or inconsistencies.
- Output only the final best-consistent structure that includes:
  - Folder layout (tree format).
  - Key files and short descriptions.
  - API routes (with method and purpose).
  - Data models and storage choice (with short rationales for why each option was selected).
  - Services and background jobs.
  - Config, logging, and observability.
  - Testing strategy and CI notes.
  - Deployment and scalaibilty notes.
- Assumption: It's safe to assume that the final structure is the best possible solution for the problem at hand. However, it may not be the only or the best possible solution.

Here are some

# The best response was the first one, because it actually proposes a concrete backend code structure with a folder tree models routes services tests and key files with purposes

## Solution Development

In [ ]:
## YOUR TURN
# Alternative 1
# STRUCTURE THE CODE AS ABOVE
# TEST WITH 2-3 DIFFERENT PROMPT ENGINEERING TECHNIQUES
# NOTE: TEST WITH VARIATIONS OF MODEL PARAMTERS; FIND THE CONFIGURATION THAT DELIVERS THE BEST OUTPUT

In [ ]:
# --- Step 1: Define raw User Prompt ---
# Note: ADJUST for your solution scenario

USER_PROMPT = "Propose a backend code structure for an AI-powered exam studying app for SAT, GRE, and Medical Exams"
print("User Prompt:\n", USER_PROMPT)

User Prompt:
 Propose a backend code structure for an AI-powered exam studying app for SAT, GRE, and Medical Exams


In [ ]:
# Technique used  Few Shot Prompting
META_PROMPT_1 = f"""
You are a Solution Design Bot.

Goal
Propose a production ready backend structure for the request below.

Examples to imitate
Example A
project
  app
    __init__.py
    models.py
    routes.py
    services.py
  tests
    test_app.py
  requirements.txt
  run.py

Example B
backend
  src
    api
      controllers.py
      serializers.py
    core
      config.py
      database.py
    features
      users.py
      exams.py
      ai_engine.py
  tests
    unit
    integration
  pyproject.toml

Instructions
Use the examples only as a style guide. Adapt them to the user need.

User request
{USER_PROMPT}

Output
1 folder layout using simple indentation only
2 key files with one line purpose each
3 API surface with method and short purpose
4 data models and storage choice with one or two lines of rationale
5 services and background jobs
6 config logging and observability
7 testing plan and minimal CI notes
8 deployment and scalability notes
9 assumptions open questions and risks with quick mitigations

Style rules
be concise and specific
use clear section headers
do not print hidden reasoning
""".strip()

In [ ]:
# Technique used  Chain of Thought with Reflection
META_PROMPT_2 = f"""
You are a Solution Design Bot.

Thinking mode
Perform deep reasoning silently. Do not reveal your chain of thought.

Two pass process
Pass one  planning
  break the problem into parts  data layer  APIs  services  security  tests  deployment
  consider at least two design options internally
  pick the most practical and extensible option
Pass two  reflection
  verify coverage of all required parts
  check consistency scalability feasibility and security
  fix any gaps before producing the answer

User request
{USER_PROMPT}

Output
Provide only the final result with
1 folder layout using indentation
2 key files and purpose
3 API routes with method and goal
4 data models and relations plus storage choice and short rationale
5 services tasks and background workers
6 config logging metrics tracing and alerts
7 testing strategy with unit integration and smoke tests plus CI notes
8 deployment runbook containers env vars secrets and rollout
9 assumptions open questions risks and simple mitigations

Quality checklist to run silently
coverage  security  fault handling  observability  migration path  first release viability
""".strip()

In [ ]:
# Technique used  Self Consistency
META_PROMPT_3 = f"""
You are a Solution Design Bot.

Method
Generate several candidate backend designs internally and compare them. Do not show the candidates. Select and present only the best consistent plan.

Internal scoring to run silently
score each candidate from 1 to 5 on
  coverage of requirements
  cohesion of modules and clarity of boundaries
  feasibility for a small first release
  scalability path and cost awareness
  security and data protection
choose the highest score candidate or merge parts to maximize the score

User request
{USER_PROMPT}

Output
Return only the consolidated best design with
1 folder layout
2 key files and purpose
3 API routes with method and intent
4 data models storage choice and rationale
5 services jobs and queues
6 config logging metrics tracing and alerting
7 testing and CI notes
8 deployment strategy with scalability tips
9 assumptions open questions risks and mitigations

Tone and format
concise professional clear headers no hidden notes
""".strip()

In [ ]:
AUGMENTED_PROMPT_1 = META_PROMPT_1
print("\n--- Augmented Prompt 1---\n")
print(AUGMENTED_PROMPT_1)


--- Augmented Prompt 1---

You are a Solution Design Bot.

Goal
Propose a production ready backend structure for the request below.

Examples to imitate
Example A
project
  app
    __init__.py
    models.py
    routes.py
    services.py
  tests
    test_app.py
  requirements.txt
  run.py

Example B
backend
  src
    api
      controllers.py
      serializers.py
    core
      config.py
      database.py
    features
      users.py
      exams.py
      ai_engine.py
  tests
    unit
    integration
  pyproject.toml

Instructions
Use the examples only as a style guide. Adapt them to the user need.

User request
Propose a backend code structure for an AI-powered exam studying app for SAT, GRE, and Medical Exams

Output
1 folder layout using simple indentation only
2 key files with one line purpose each
3 API surface with method and short purpose
4 data models and storage choice with one or two lines of rationale
5 services and background jobs
6 config logging and observability
7 testing p

In [ ]:
AUGMENTED_PROMPT_2 = META_PROMPT_2
print("\n--- Augmented Prompt 2---\n")
print(AUGMENTED_PROMPT_2)


--- Augmented Prompt 2---

You are a Solution Design Bot.

Thinking mode
Perform deep reasoning silently. Do not reveal your chain of thought.

Two pass process
Pass one  planning
  break the problem into parts  data layer  APIs  services  security  tests  deployment
  consider at least two design options internally
  pick the most practical and extensible option
Pass two  reflection
  verify coverage of all required parts
  check consistency scalability feasibility and security
  fix any gaps before producing the answer

User request
Propose a backend code structure for an AI-powered exam studying app for SAT, GRE, and Medical Exams

Output
Provide only the final result with
1 folder layout using indentation
2 key files and purpose
3 API routes with method and goal
4 data models and relations plus storage choice and short rationale
5 services tasks and background workers
6 config logging metrics tracing and alerts
7 testing strategy with unit integration and smoke tests plus CI notes

In [ ]:
AUGMENTED_PROMPT_3 = META_PROMPT_3
print("\n--- Augmented Prompt 3---\n")
print(AUGMENTED_PROMPT_3)


--- Augmented Prompt 3---

You are a Solution Design Bot.

Method
Generate several candidate backend designs internally and compare them. Do not show the candidates. Select and present only the best consistent plan.

Internal scoring to run silently
score each candidate from 1 to 5 on
  coverage of requirements
  cohesion of modules and clarity of boundaries
  feasibility for a small first release
  scalability path and cost awareness
  security and data protection
choose the highest score candidate or merge parts to maximize the score

User request
Propose a backend code structure for an AI-powered exam studying app for SAT, GRE, and Medical Exams

Output
Return only the consolidated best design with
1 folder layout
2 key files and purpose
3 API routes with method and intent
4 data models storage choice and rationale
5 services jobs and queues
6 config logging metrics tracing and alerting
7 testing and CI notes
8 deployment strategy with scalability tips
9 assumptions open questions 

In [ ]:
response, tokens = model_request(
    prompt=AUGMENTED_PROMPT_1,   # Few Shot
    model=DEFAULT_MODEL,
    temperature=0.35,            # low for format fidelity
    top_k=30,
    top_p=0.88,
    num_predict=600,
    context_window=2048
)

In [ ]:
print("\n--- Bot Response ---\n")
print(response)
print(f"\n[Tokens used: {tokens}]")


--- Bot Response ---

Example A: Production Ready Backend Structure

Project
   app
     __init__.py
     models.py
     routes.py
     services.py
   tests
     test_app.py
   requirements.txt
   run.py

Backend
   src
     api
       controller.py
       serializers.py
     core
       config.py
       database.py
     features
       users.py
       exams.py
       ai_engine.py

API surface:
1. A single-page API endpoint for each feature, with a GET and POST request method.
2. The endpoint should return JSON data, including the feature name and its corresponding details.
3. Each endpoint should have a clear purpose statement, such as "Get users for SAHARA exam", or "Create an AI-powered exam studying app".
4. Use descriptive variable names to make it easy for developers to understand what each endpoint does.
5. Ensure that all endpoints are documented with Swagger UI and have appropriate comments explaining the purpose of each endpoint.
6. Add error handling in case of unexpected H

In [ ]:
response, tokens = model_request(
    prompt=AUGMENTED_PROMPT_2,   # CoT with Reflection
    model=DEFAULT_MODEL,
    temperature=0.6,             # moderate for better reasoning
    top_k=40,
    top_p=0.95,
    num_predict=900,             # CoT answers are longer
    context_window=2048
)

print("\n--- Bot Response ---\n")
print(response)
print(f"\n[Tokens used: {tokens}]")


--- Bot Response ---

User request: Provide a backend code structure for an AI-powered exam studying app for SA, GRE, and Medical Exams. The solution should have a folder layout using indentation, key files and purpose, API routes with method and goal, data models and relations plus storage choice and short rationales, services tasks and background workers, config logging metrics tracing and alerts, testing strategy with unit integration and smoke tests plus CI notes, deployment runbook containers env vars secret, and risk mitigation strategies. The solution should be silent and perform deep reasoning without revealing the chain of thought.

[Tokens used: 126]


In [ ]:
def run_self_consistent(prompt, n=5):
    candidates = []
    for _ in range(n):
        resp, _ = model_request(
            prompt=prompt,       # Self Consistency
            model=DEFAULT_MODEL,
            temperature=0.9,     # encourage diverse candidates
            top_k=50,
            top_p=0.97,
            num_predict=900,
            context_window=2048
        )
        candidates.append(resp)

    headers = [
        "Folder layout",
        "Key files",
        "API",
        "Data models",
        "Services",
        "Config",
        "Testing",
        "Deployment",
        "Assumptions"
    ]
    def score(txt):
        return sum(h in txt for h in headers)

    return max(candidates, key=score)

response = run_self_consistent(AUGMENTED_PROMPT_3, n=5)
print("\n--- Bot Response ---\n")
print(response)



--- Bot Response ---

Method:
1. Generate candidate backend designs internally and compare them based on internal scoring for each criterion as described in the context.
2. Score each candidate from 1 to 5 on each criterion, with 5 being the highest score.
3. Choose or merge the highest-scoring candidate with minimal overlap.
4. Provide a consistent plan to the client that presents only the most favourable design.
5. Document and deliver all the above.

Internal scoring:
1. Coverage of requirements
    - Include as much detailed information about the requirements of each subsystem or component as possible, highlighting gaps and inconsistencies.
    - Consider the needs of different target audiences and focus on providing information that is relevant to them.
2. Cohesion of modules and clarity of boundaries
    - Ensure a clear and coherent integration of subsystems or components into one logical entity that meets the requirements specified in each design phase.
    - Focus on breaking

# The first one was the best, because it gives a concrete folder tree, key files, API surface, data models, services, tests, and run script

---

Part of my GenAI portfolio. For the packaged library, tests, and the rest of the project, see the [README](../README.md).